# Day 01 — Excel → GIS Update Automation

## Problem

A GIS team receives recurring Excel records that must be matched to existing GIS features. A manual workflow usually involves joins, field updates, review of unmatched IDs, and repetitive QA.

## Goal

Build a reproducible workflow that:

- reads an Excel workbook,
- validates required columns,
- checks for null and duplicate IDs,
- converts the portable GeoJSON sample into a local geodatabase feature class,
- matches Excel records to GIS features,
- updates selected GIS attributes,
- reports unmatched records,
- tests failure cases,
- and produces a processing summary.

## Developer mindset

**Validate first → edit second.**

The notebook uses only synthetic data and is safe to publish.

## 0. Before you run the notebook

Start Jupyter **from the Day 01 project folder**:

```text
C:\Users\<your-user>\Downloads\GitHub\gis-developer-30-day-lab\day-01-excel-gis-update
```

Then open this notebook.

The notebook intentionally uses **relative paths** so it remains portable and publishable.

In [1]:
from pathlib import Path
import json
import pandas as pd
import arcpy

PROJECT_DIR = Path.cwd().resolve()

print("Current working directory:")
print(PROJECT_DIR)

EXPECTED_FOLDER = "day-01-excel-gis-update"

if PROJECT_DIR.name != EXPECTED_FOLDER:
    raise RuntimeError(
        f"\nThis notebook should be launched from the '{EXPECTED_FOLDER}' folder.\n"
        f"Current folder: {PROJECT_DIR}\n\n"
        "Close Jupyter, open Command Prompt, cd into the Day 01 folder, "
        "and launch Jupyter again."
    )

print("\nWorking directory check: PASS")

Current working directory:
C:\Users\dshrestha\Downloads\GitHub\gis-developer-30-day-lab\day-01-excel-gis-update

Working directory check: PASS


## 1. Define project paths

We keep source/sample data separate from generated working data.

- `data/sample/` → committed/reproducible source data
- `output/` → generated local outputs
- `src/` → reusable Python module

In [2]:
DATA_DIR = PROJECT_DIR / "data" / "sample"
OUTPUT_DIR = PROJECT_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

EXCEL_PATH = DATA_DIR / "incoming_updates.xlsx"
GEOJSON_PATH = DATA_DIR / "infrastructure_assets.geojson"

print("Excel exists:  ", EXCEL_PATH.exists(), EXCEL_PATH)
print("GeoJSON exists:", GEOJSON_PATH.exists(), GEOJSON_PATH)

if not EXCEL_PATH.exists():
    raise FileNotFoundError(
        f"{EXCEL_PATH}\nRun data/create_sample_data.py first."
    )

if not GEOJSON_PATH.exists():
    raise FileNotFoundError(
        f"{GEOJSON_PATH}\nRun data/create_sample_data.py first."
    )

Excel exists:   True C:\Users\dshrestha\Downloads\GitHub\gis-developer-30-day-lab\day-01-excel-gis-update\data\sample\incoming_updates.xlsx
GeoJSON exists: True C:\Users\dshrestha\Downloads\GitHub\gis-developer-30-day-lab\day-01-excel-gis-update\data\sample\infrastructure_assets.geojson


## 2. Inspect the Excel workbook

Before writing update logic, inspect what the incoming workbook actually contains.

In [3]:
excel_file = pd.ExcelFile(EXCEL_PATH)

print("Workbook sheets:")
for sheet in excel_file.sheet_names:
    print(" -", sheet)

Workbook sheets:
 - updates
 - duplicate_id_test
 - null_id_test
 - missing_column_test


In [4]:
df = pd.read_excel(EXCEL_PATH, sheet_name="updates")

print("Rows:", len(df))
print("Columns:", list(df.columns))

df

Rows: 5
Columns: ['asset_id', 'status_new', 'priority_new', 'inspector_new', 'inspection_date']


,asset_id,status_new,priority_new,inspector_new,inspection_date
0,AST-001,Active,High,L. Nguyen,2026-09-20
1,AST-003,Maintenance,High,L. Nguyen,2026-09-21
2,AST-005,Active,Medium,R. Gomez,2026-09-22
3,AST-007,Active,High,R. Gomez,2026-09-23
4,AST-999,Active,Low,R. Gomez,2026-09-24


### What should you notice?

The normal update sheet contains:

- `asset_id` → join key
- `status_new`
- `priority_new`
- `inspector_new`
- `inspection_date`

`AST-999` is intentional. It does **not** exist in the GIS dataset and should be reported as unmatched rather than silently discarded.

## 3. Validate required columns

A production workflow should fail **before GIS editing begins** when the input schema is wrong.

In [5]:
REQUIRED_COLUMNS = {
    "asset_id",
    "status_new",
    "priority_new",
    "inspector_new",
    "inspection_date",
}

def validate_required_columns(dataframe, required_columns):
    """Raise ValueError when one or more required columns are missing."""
    actual_columns = set(dataframe.columns)
    missing = set(required_columns) - actual_columns

    if missing:
        raise ValueError(
            "Missing required column(s): "
            + ", ".join(sorted(missing))
        )

    return True

validate_required_columns(df, REQUIRED_COLUMNS)

True

## 4. Validate the unique ID field

The update workflow depends on `asset_id` being usable as a stable join key.

We will reject:

- null IDs,
- duplicate incoming IDs.

In [6]:
def validate_ids(dataframe, id_field="asset_id"):
    """Validate that incoming IDs are non-null and unique."""

    null_mask = dataframe[id_field].isna()
    if null_mask.any():
        bad_rows = dataframe.index[null_mask].tolist()
        raise ValueError(
            f"Null {id_field} value(s) found at DataFrame row index(es): {bad_rows}"
        )

    duplicate_mask = dataframe[id_field].duplicated(keep=False)
    if duplicate_mask.any():
        duplicates = sorted(
            dataframe.loc[duplicate_mask, id_field].astype(str).unique().tolist()
        )
        raise ValueError(
            f"Duplicate {id_field} value(s): {duplicates}"
        )

    return True

validate_ids(df)

True

## 5. Test the validation against controlled failure cases

A validator is only useful if we prove that it catches bad inputs.

In [7]:
def validate_update_sheet(dataframe):
    validate_required_columns(dataframe, REQUIRED_COLUMNS)
    validate_ids(dataframe, "asset_id")
    return True

test_sheets = [
    "duplicate_id_test",
    "null_id_test",
    "missing_column_test",
]

for sheet in test_sheets:
    test_df = pd.read_excel(EXCEL_PATH, sheet_name=sheet)

    print(f"\nTesting sheet: {sheet}")

    try:
        validate_update_sheet(test_df)
        print("  UNEXPECTED: validation passed")
    except ValueError as exc:
        print("  EXPECTED FAILURE:", exc)


Testing sheet: duplicate_id_test
  EXPECTED FAILURE: Duplicate asset_id value(s): ['AST-003']

Testing sheet: null_id_test
  EXPECTED FAILURE: Null asset_id value(s) found at DataFrame row index(es): [5]

Testing sheet: missing_column_test
  EXPECTED FAILURE: Missing required column(s): priority_new


Expected behavior:

- `duplicate_id_test` → duplicate `AST-003`
- `null_id_test` → null ID
- `missing_column_test` → missing `priority_new`

If those three cases fail for the expected reason, our input validation is doing its job.

## 6. Build an update lookup

Once the Excel input passes validation, convert it into a dictionary keyed by `asset_id`.

That gives us fast lookup while iterating through GIS features.

In [8]:
validate_update_sheet(df)

update_lookup = (
    df.set_index("asset_id")[
        ["status_new", "priority_new", "inspector_new", "inspection_date"]
    ]
    .to_dict(orient="index")
)

update_lookup

{'AST-001': {'status_new': 'Active',
  'priority_new': 'High',
  'inspector_new': 'L. Nguyen',
  'inspection_date': '2026-09-20'},
 'AST-003': {'status_new': 'Maintenance',
  'priority_new': 'High',
  'inspector_new': 'L. Nguyen',
  'inspection_date': '2026-09-21'},
 'AST-005': {'status_new': 'Active',
  'priority_new': 'Medium',
  'inspector_new': 'R. Gomez',
  'inspection_date': '2026-09-22'},
 'AST-007': {'status_new': 'Active',
  'priority_new': 'High',
  'inspector_new': 'R. Gomez',
  'inspection_date': '2026-09-23'},
 'AST-999': {'status_new': 'Active',
  'priority_new': 'Low',
  'inspector_new': 'R. Gomez',
  'inspection_date': '2026-09-24'}}

## 7. Create a local working geodatabase

The repository stores GIS source data as GeoJSON because it is small and portable.

For ArcGIS editing, convert it into a **local file geodatabase feature class**.

This notebook creates:

```text
output/Day01_Working.gdb
```

The `output/` folder is not intended to be committed.

In [9]:
GDB_PATH = OUTPUT_DIR / "Day01_Working.gdb"
FC_NAME = "infrastructure_assets"
FC_PATH = GDB_PATH / FC_NAME

if not arcpy.Exists(str(GDB_PATH)):
    arcpy.management.CreateFileGDB(
        str(OUTPUT_DIR),
        GDB_PATH.name
    )

if arcpy.Exists(str(FC_PATH)):
    arcpy.management.Delete(str(FC_PATH))

arcpy.conversion.JSONToFeatures(
    str(GEOJSON_PATH),
    str(FC_PATH),
    "POINT"
)

print("Working feature class created:")
print(FC_PATH)
print("Feature count:", int(arcpy.management.GetCount(str(FC_PATH))[0]))

Working feature class created:
C:\Users\dshrestha\Downloads\GitHub\gis-developer-30-day-lab\day-01-excel-gis-update\output\Day01_Working.gdb\infrastructure_assets
Feature count: 8


## 8. Validate the GIS schema

We should also validate the GIS side before editing.

In [10]:
GIS_REQUIRED_FIELDS = {
    "asset_id",
    "status",
    "priority",
    "inspector",
    "last_inspection",
}

gis_fields = {field.name for field in arcpy.ListFields(str(FC_PATH))}
missing_gis_fields = GIS_REQUIRED_FIELDS - gis_fields

print("GIS fields:")
print(sorted(gis_fields))

if missing_gis_fields:
    raise ValueError(
        "GIS dataset is missing required field(s): "
        + ", ".join(sorted(missing_gis_fields))
    )

print("\nGIS schema validation: PASS")

GIS fields:
['OBJECTID', 'Shape', 'asset_id', 'asset_type', 'inspector', 'last_inspection', 'priority', 'status']

GIS schema validation: PASS


## 9. Inspect GIS records before editing

In [11]:
preview_fields = [
    "asset_id",
    "asset_type",
    "status",
    "priority",
    "inspector",
    "last_inspection",
]

rows_before = []

with arcpy.da.SearchCursor(str(FC_PATH), preview_fields) as cursor:
    for row in cursor:
        rows_before.append(row)

before_df = pd.DataFrame(rows_before, columns=preview_fields)
before_df

,asset_id,asset_type,status,priority,inspector,last_inspection
0,AST-001,Valve,Active,Medium,A. Rivera,2026-07-15
1,AST-002,Valve,Active,Low,A. Rivera,2026-07-18
2,AST-003,Meter,Active,Medium,J. Chen,2026-06-29
3,AST-004,Regulator,Active,High,J. Chen,2026-06-20
4,AST-005,Meter,Inactive,Low,M. Patel,2026-05-11
5,AST-006,Valve,Active,Medium,M. Patel,2026-08-02
6,AST-007,Regulator,Active,Low,S. Brooks,2026-08-10
7,AST-008,Meter,Active,Medium,S. Brooks,2026-08-17


## 10. Compare incoming IDs with GIS IDs before editing

This lets us know what will match before touching the data.

In [12]:
gis_ids = set()

with arcpy.da.SearchCursor(str(FC_PATH), ["asset_id"]) as cursor:
    for (asset_id,) in cursor:
        gis_ids.add(asset_id)

incoming_ids = set(df["asset_id"])

matched_ids = sorted(incoming_ids & gis_ids)
unmatched_ids = sorted(incoming_ids - gis_ids)

print("Matched IDs:", matched_ids)
print("Unmatched IDs:", unmatched_ids)
print("Expected unmatched ID: AST-999")

Matched IDs: ['AST-001', 'AST-003', 'AST-005', 'AST-007']
Unmatched IDs: ['AST-999']
Expected unmatched ID: AST-999


## 11. Update matched GIS features

Field mapping:

| Excel | GIS |
|---|---|
| `status_new` | `status` |
| `priority_new` | `priority` |
| `inspector_new` | `inspector` |
| `inspection_date` | `last_inspection` |

Only existing GIS IDs are updated.

In [13]:
update_fields = [
    "asset_id",
    "status",
    "priority",
    "inspector",
    "last_inspection",
]

updated_count = 0
updated_ids = []

with arcpy.da.UpdateCursor(str(FC_PATH), update_fields) as cursor:
    for row in cursor:
        asset_id = row[0]

        if asset_id not in update_lookup:
            continue

        update = update_lookup[asset_id]

        row[1] = update["status_new"]
        row[2] = update["priority_new"]
        row[3] = update["inspector_new"]
        row[4] = update["inspection_date"]

        cursor.updateRow(row)

        updated_count += 1
        updated_ids.append(asset_id)

print("Updated feature count:", updated_count)
print("Updated IDs:", sorted(updated_ids))

Updated feature count: 4
Updated IDs: ['AST-001', 'AST-003', 'AST-005', 'AST-007']


## 12. Verify the edits

Never assume an edit worked merely because the code ran.

In [14]:
rows_after = []

with arcpy.da.SearchCursor(str(FC_PATH), preview_fields) as cursor:
    for row in cursor:
        rows_after.append(row)

after_df = pd.DataFrame(rows_after, columns=preview_fields)
after_df

,asset_id,asset_type,status,priority,inspector,last_inspection
0,AST-001,Valve,Active,High,L. Nguyen,2026-09-20
1,AST-002,Valve,Active,Low,A. Rivera,2026-07-18
2,AST-003,Meter,Maintenance,High,L. Nguyen,2026-09-21
3,AST-004,Regulator,Active,High,J. Chen,2026-06-20
4,AST-005,Meter,Active,Medium,R. Gomez,2026-09-22
5,AST-006,Valve,Active,Medium,M. Patel,2026-08-02
6,AST-007,Regulator,Active,High,R. Gomez,2026-09-23
7,AST-008,Meter,Active,Medium,S. Brooks,2026-08-17


## 13. Write the unmatched-record report

In [15]:
unmatched_df = df[df["asset_id"].isin(unmatched_ids)].copy()

UNMATCHED_REPORT = OUTPUT_DIR / "unmatched_records.csv"
unmatched_df.to_csv(UNMATCHED_REPORT, index=False)

print("Unmatched report:")
print(UNMATCHED_REPORT)

unmatched_df

Unmatched report:
C:\Users\dshrestha\Downloads\GitHub\gis-developer-30-day-lab\day-01-excel-gis-update\output\unmatched_records.csv


,asset_id,status_new,priority_new,inspector_new,inspection_date
4,AST-999,Active,Low,R. Gomez,2026-09-24


## 14. Create a processing summary

In [16]:
summary = {
    "incoming_rows": len(df),
    "gis_features": int(arcpy.management.GetCount(str(FC_PATH))[0]),
    "matched_incoming_ids": len(matched_ids),
    "updated_features": updated_count,
    "unmatched_incoming_ids": unmatched_ids,
}

SUMMARY_PATH = OUTPUT_DIR / "processing_summary.json"

with SUMMARY_PATH.open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("\nSummary written to:")
print(SUMMARY_PATH)

{
  "incoming_rows": 5,
  "gis_features": 8,
  "matched_incoming_ids": 4,
  "updated_features": 4,
  "unmatched_incoming_ids": [
    "AST-999"
  ]
}

Summary written to:
C:\Users\dshrestha\Downloads\GitHub\gis-developer-30-day-lab\day-01-excel-gis-update\output\processing_summary.json


## 15. Day 01 validation checklist

Version 1 is complete when you can confirm:

- [ ] sample data generated successfully
- [ ] Excel required columns validated
- [ ] null IDs rejected
- [ ] duplicate IDs rejected
- [ ] missing columns rejected
- [ ] GeoJSON converted to a local feature class
- [ ] GIS schema validated
- [ ] matched records updated correctly
- [ ] `AST-999` reported as unmatched
- [ ] processing summary created
- [ ] notebook runs top-to-bottom from a clean restart

## 16. Git checkpoint

After the notebook runs successfully, return to Command Prompt:

```bat
git status
```

Do **not** commit generated files from `output/`.

Then add the notebook and documentation changes:

```bat
git add Day01_Excel_to_GIS_Update_Automation.ipynb
git add README.md
git status
```

Commit:

```bat
git commit -m "feat: complete Day 01 Excel to GIS update workflow"
```

Push:

```bat
git push
```

We will review the branch diff before creating the pull request.

## Interview explanation

A concise way to explain this exercise:

> I built a validated Excel-to-GIS update workflow that checks incoming schema and unique IDs before editing, converts portable source data into a local geodatabase, matches records by stable asset ID, updates GIS attributes with ArcPy, reports unmatched records, tests controlled failure cases, and generates an auditable processing summary.

That is the difference between **performing a join manually** and **engineering a repeatable GIS update process**.